# Synergy Analysis

In [ ]:
import scanpy as sc

# Load the h5ad file
adata = sc.read_h5ad('/gpfs/radev/home/dfl32/project/C2SV2-PERT/percept_data/all.h5ad') # Replace with the new data path
print(adata)
adata.obs

In [ ]:
# Group by sample_type and sample_id
grouped = adata.obs.groupby(['sample_type', 'sample_id']).size()
print(grouped)


In [ ]:
yumisty_adata = adata[adata.obs['sample_id'] == 'YUMISTY'].copy()
yumisty_adata.obs['stim'].value_counts()

In [ ]:
import pandas as pd
import numpy as np
import cinemaot as co
import matplotlib.colors as colors
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 250

perturbationlist = list(set(yumisty_adata.obs['stim'].values.tolist()))
response = yumisty_adata[yumisty_adata.obs['stim'].isin(['NS']),:].copy()
print(response)
for i in perturbationlist:
    if i != 'NS':
        yumisty_adata_ = yumisty_adata[yumisty_adata.obs['stim'].isin(['NS',i]),:].copy()
        sc.pp.pca(yumisty_adata_)
        cf, ot, de = co.cinemaot.cinemaot_unweighted(yumisty_adata_,obs_label='stim', ref_label='NS', expr_label=i,mode='parametric',thres=0.5,smoothness=1e-3,eps=1e-2,preweight_label='cell_category')
        subset_mock = yumisty_adata_[yumisty_adata_.obs['stim'].isin(['NS']),:]
        de_c = sc.AnnData(subset_mock.X.toarray()-de.X)
        de_c.obs['cell_category'] = subset_mock.obs['cell_category'].values
        de_c.obs['stim'] = i
        de_c.var_names = subset_mock.var_names
        print(de_c)
        response = sc.concat([response,de_c])

sc.pp.pca(response)
sc.pp.neighbors(response)
sc.tl.umap(response)
synergylist = ['IFNb', 'IFNb + aPD1', 'aPD1', 'No stimulation']
combolist = ['IFNb + aPD1']
synergy = response[response.obs['stim'].isin(combolist),:].copy()
combo = response[response.obs['stim'].isin(combolist),:].copy()

combo.X[combo.obs['stim']=='IFNb + aPD1',:] = response.X[response.obs['stim']=='IFNb + aPD1',:]-response.X[response.obs['stim']=='IFNb',:] - response.X[response.obs['stim']=='aPD1',:] + response.X[response.obs['stim']=='NS',:]


combo

In [ ]:
print(combo)
# Convert sparse matrix to dense for boolean indexing
combo_dense = combo.X.toarray()
mean_values = np.mean(combo_dense, axis=0)
high_synergy_genes = combo_dense[:, np.abs(mean_values) > 0.15]
print(f"Selected {high_synergy_genes.shape[1]} genes with high synergy scores")
high_synergy_genes

In [ ]:
# Convert sparse matrix to dense for boolean indexing
combo_dense = combo.X.toarray()
mean_values = np.mean(combo_dense, axis=0)
high_synergy_genes = combo_dense[:, np.abs(mean_values) > 0.15]
np.linalg.norm(high_synergy_genes, axis=1)

In [ ]:
sc.pp.pca(combo)
sc.pp.neighbors(combo)
sc.tl.umap(combo)

In [ ]:
import matplotlib


means = combo.X.toarray()[:,np.abs(np.mean(combo.X.toarray(),axis=0))>0.15]

synergyscore = np.linalg.norm(means, axis=1)
sc.pl.umap(combo,color=['cell_category'],wspace=0.5,size=0.005*synergyscore**2)

In [ ]:
combo.obs['synergy'] = 0.01*synergyscore
combo_filtered = combo[~combo.obs['cell_category'].isin(['MEL_TUMOR', 'MYELOID'])].copy()
sc.pl.violin(combo_filtered,keys=['synergy'],groupby='cell_category',rotation=90)

In [ ]:
tumor_adata = adata[adata.obs['sample_type'].isin(['MEL', 'MCC'])].copy()
tumor_adata

In [ ]:
import pandas as pd
import numpy as np
import cinemaot as co
import matplotlib.colors as colors
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 250

perturbationlist = list(set(tumor_adata.obs['stim'].values.tolist()))
tumor_response = tumor_adata[tumor_adata.obs['stim'].isin(['NS']),:].copy()
for i in perturbationlist:
    if i != 'NS':
        tumor_adata_ = tumor_adata[tumor_adata.obs['stim'].isin(['NS',i]),:].copy()
        sc.pp.pca(tumor_adata_)
        cf, ot, de = co.cinemaot.cinemaot_unweighted(tumor_adata_,obs_label='stim', ref_label='NS', expr_label=i,mode='parametric',thres=0.5,smoothness=1e-3,eps=1e-2,preweight_label='cell_category')
        subset_mock = tumor_adata_[tumor_adata_.obs['stim'].isin(['NS']),:]
        de_c = sc.AnnData(subset_mock.X.toarray()-de.X)
        de_c.obs['cell_category'] = subset_mock.obs['cell_category'].values
        de_c.obs['stim'] = i
        de_c.var_names = subset_mock.var_names
        tumor_response = sc.concat([tumor_response,de_c])

sc.pp.pca(tumor_response)
sc.pp.neighbors(tumor_response)
sc.tl.umap(tumor_response)
synergylist = ['IFNb', 'IFNb + aPD1', 'aPD1', 'No stimulation']
combolist = ['IFNb + aPD1']
tumor_combo = tumor_response[tumor_response.obs['stim'].isin(combolist),:].copy()

tumor_combo.X[tumor_combo.obs['stim']=='IFNb + aPD1',:] = tumor_response.X[tumor_response.obs['stim']=='IFNb + aPD1',:]-tumor_response.X[tumor_response.obs['stim']=='IFNb',:] - tumor_response.X[tumor_response.obs['stim']=='aPD1',:] + tumor_response.X[tumor_response.obs['stim']=='NS',:]


tumor_combo

In [ ]:
sc.pp.pca(tumor_combo)
sc.pp.neighbors(tumor_combo)
sc.tl.umap(tumor_combo)
tumor_means = tumor_combo.X.toarray()[:,np.abs(np.mean(tumor_combo.X.toarray(),axis=0))>0.15]
tumor_synergyscore = np.linalg.norm(tumor_means, axis=1)
sc.pl.umap(tumor_combo,color=['cell_category'],wspace=0.5,size=0.005*tumor_synergyscore**1.5)

In [ ]:
tumor_combo.obs['synergy'] = 0.01*tumor_synergyscore
sc.pl.violin(tumor_combo,keys=['synergy'],groupby='cell_category',rotation=90)

# MDK leave-one-out: responders vs non-responders

CINEMA-OT with response as the contrast (ref = NR, expr = R), run separately for each treatment given to all 4 MCC patients, leaving out one MCC patient at a time.

In [ ]:
import scanpy as sc

# Load the h5ad file
adata = sc.read_h5ad('/gpfs/radev/home/dfl32/project/C2SV2-PERT/percept_data/all.h5ad') # Replace with the new data path
adata.obs

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cinemaot as co
plt.rcParams['figure.dpi'] = 250

# MCC patients (paper labels); MCC1 and MCC4 responded, MCC2 and MCC3 did not
mcc_names = {'MCC79': 'MCC1', 'YUEYE': 'MCC2', 'YURAD': 'MCC3', 'YUMISTY': 'MCC4'}
responders = ['MCC79', 'YUMISTY']

mcc = adata[adata.obs['sample_id'].isin(list(mcc_names))].copy()
# X is raw counts; CINEMA-OT expects normalized, log-transformed input
sc.pp.normalize_total(mcc, target_sum=1e4)
sc.pp.log1p(mcc)
mcc.obs['response'] = pd.Categorical(np.where(mcc.obs['sample_id'].isin(responders), 'R', 'NR'), categories=['NR', 'R'])

# Treatments given to all 4 MCC patients, with the unstimulated control (NS) first
counts = mcc.obs.groupby(['sample_id', 'stim'], observed=True).size().unstack(fill_value=0)
treatments = sorted((s for s in counts.columns if (counts[s] > 0).all()), key=lambda s: s != 'NS')

# One run per left-out patient, in the order of the four comparisons
comparisons = {}
for left_out in ['YUMISTY', 'MCC79', 'YURAD', 'YUEYE']:
    kept = sorted((s for s in mcc_names if s != left_out), key=mcc_names.get)
    r = ' + '.join(mcc_names[s] for s in kept if s in responders)
    nr = ' + '.join(mcc_names[s] for s in kept if s not in responders)
    comparisons[left_out] = f'{r} (R) vs {nr} (NR), without {mcc_names[left_out]}'
print('treatments:', treatments)
print(*comparisons.values(), sep='\n')
counts

In [ ]:
# For each left-out patient and each treatment, CINEMA-OT removes the confounders between
# responder and non-responder cells; de has one row per non-responder cell
des = {}
for left_out in comparisons:
    for stim in treatments:
        sub = mcc[(mcc.obs['sample_id'] != left_out) & (mcc.obs['stim'] == stim)].copy()
        sc.pp.pca(sub)
        cf, ot, de = co.cinemaot.cinemaot_unweighted(sub, obs_label='response', ref_label='NR', expr_label='R', mode='parametric', thres=0.5, smoothness=1e-3, eps=1e-2, preweight_label='cell_category')
        # cinemaot returns NR − matched R; flip so positive = higher in responders
        de.X = -de.X
        des[left_out, stim] = de
        print(f'{stim}, {comparisons[left_out]}: {de.n_obs} NR cells')

In [ ]:
def assignleiden(adata, ctobs, clobs, label):
    # Same as cinemaot.utils.assignleiden (that module needs plotly, which isn't installed):
    # name each cluster after its dominant cell type, numbered within that type
    n_seen, names = {}, {}
    for cl in adata.obs[clobs].cat.categories:
        ct = adata.obs.loc[adata.obs[clobs] == cl, ctobs].value_counts().idxmax()
        names[cl] = f'{ct}{n_seen.get(ct, 0)}'
        n_seen[ct] = n_seen.get(ct, 0) + 1
    adata.obs[label] = adata.obs[clobs].map(names).astype(str)

for (left_out, stim), de in des.items():
    sc.pp.pca(de)
    sc.pp.neighbors(de, use_rep='X_embedding')
    sc.tl.leiden(de, resolution=0.2)
    sc.tl.umap(de)
    assignleiden(de, 'cell_category', 'leiden', 'typebatch')
    keys = ['typebatch', 'cell_category', 'sample_id']
    sc.pl.umap(de, color=keys, title=[f'{k}: {stim}, without {mcc_names[left_out]}' for k in keys], wspace=0.5)

In [ ]:
# Tutorial view: DEGs of each de cluster vs. the other clusters
for (left_out, stim), de in des.items():
    sc.tl.rank_genes_groups(de, 'typebatch', method='t-test')
    sc.pl.rank_genes_groups(de, n_genes=25, sharey=False, show=False)
    plt.suptitle(f'{stim}: {comparisons[left_out]}', y=1.02)
    plt.show()

In [ ]:
# Responder-vs-non-responder DEGs: genes ranked by mean de over all non-responder cells
top = {}
for (left_out, stim), de in des.items():
    mean = pd.Series(np.asarray(de.X.mean(axis=0)).ravel(), index=de.var_names).sort_values(ascending=False)
    top[comparisons[left_out], stim] = {'up in R': ', '.join(mean.index[:15]), 'up in NR': ', '.join(mean.index[::-1][:15])}
pd.set_option('display.max_colwidth', None)
pd.DataFrame(top).T

In [ ]:
# Where MDK ranks by mean de (rank 1 = top of all genes), over all non-responder cells and over
# non-responder MCC tumor cells only; n_R_tumor = responder tumor cells those could be matched to
rows = []
for (left_out, stim), de in des.items():
    in_run = (mcc.obs['sample_id'] != left_out) & (mcc.obs['stim'] == stim)
    row = {'comparison': comparisons[left_out], 'treatment': stim, 'left_out': mcc_names[left_out],
           'n_R_tumor': int((in_run & (mcc.obs['response'] == 'R') & (mcc.obs['cell_category'] == 'MCC_TUMOR')).sum())}
    for name, cells in [('all', slice(None)), ('tumor', (de.obs['cell_category'] == 'MCC_TUMOR').values)]:
        mean = np.asarray(de.X[cells].mean(axis=0)).ravel()
        m = mean[de.var_names.get_loc('MDK')]
        row.update({f'n_NR_{name}': de.X[cells].shape[0], f'MDK_{name}': m,
                    f'rank_up_in_R_{name}': int((mean > m).sum()) + 1, f'rank_up_in_NR_{name}': int((mean < m).sum()) + 1})
    rows.append(row)
pd.DataFrame(rows).set_index(['comparison', 'treatment'])

In [ ]:
# MDK rank among genes higher in NR, by treatment and by left-out patient (log scale, 1 = top).
# Hollow markers: without MCC4 the only responder is MCC1, which has no tumor cells to match to
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import os
os.makedirs('percept_plots', exist_ok=True)

mdk = pd.DataFrame(rows)
n_genes = next(iter(des.values())).n_vars
samples = sorted(mcc_names.values())
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#4a3aa7']  # blue, orange, aqua, violet: colorblind-safe as a set
shapes = ['o', 's', '^', 'D']

def mdk_rank_scatter(x, hue, x_order, hue_order, xlabel, name):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    for ax, cells, title in zip(axes, ['all', 'tumor'], ['All NR cells', 'NR tumor cells only']):
        for j, h in enumerate(hue_order):
            d = mdk[mdk[hue] == h]
            xs = d[x].map(x_order.index) + (j - 1.5) * 0.12
            ys = d[f'rank_up_in_NR_{cells}']
            hollow = d['left_out'] == 'MCC4'
            ax.scatter(xs[~hollow], ys[~hollow], s=64, marker=shapes[j], c=colors[j], edgecolors='white', linewidths=1, zorder=3)
            ax.scatter(xs[hollow], ys[hollow], s=64, marker=shapes[j], facecolors='none', edgecolors=colors[j], linewidths=1.5, zorder=3)
        ax.axhline(n_genes / 100, color='#8a8984', lw=1, ls='--', zorder=1)
        ax.text(len(x_order) - 0.55, n_genes / 100 / 1.15, 'top 1%', color='#52514e', fontsize=8, ha='right', va='bottom')
        ax.set_yscale('log')
        ax.set_ylim(n_genes * 1.5, 0.7)
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:,.0f}'))
        ax.set_xticks(range(len(x_order)), x_order)
        ax.set_xlim(-0.5, len(x_order) - 0.5)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel(xlabel)
        ax.grid(axis='y', color='#e5e4df', lw=0.8)
        ax.set_axisbelow(True)
        ax.spines[['top', 'right']].set_visible(False)
    axes[0].set_ylabel(f'MDK rank among genes higher in NR\n(1 = top of {n_genes:,})')
    handles = [Line2D([], [], ls='', marker=shapes[j], markersize=8, markeredgecolor=colors[j] if hue == 'left_out' and h == 'MCC4' else 'white',
                      markerfacecolor='none' if hue == 'left_out' and h == 'MCC4' else colors[j], label=h) for j, h in enumerate(hue_order)]
    handles.append(Line2D([], [], ls='', marker='o', markersize=8, markerfacecolor='none', markeredgecolor='#52514e', label='hollow: without MCC4\n(no R tumor cells)'))
    fig.legend(handles=handles, title='Left out' if hue == 'left_out' else 'Treatment', loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)
    fig.tight_layout()
    for ext in ('pdf', 'png'):
        fig.savefig(f'percept_plots/{name}.{ext}', bbox_inches='tight')
    plt.show()

mdk_rank_scatter('treatment', 'left_out', treatments, samples, 'Treatment', 'mdk_rank_by_treatment')
mdk_rank_scatter('left_out', 'treatment', samples, treatments, 'Left-out patient', 'mdk_rank_by_left_out')